# 02 · SQL Analysis

The same business questions as notebook 01, answered in **SQL** with [DuckDB](https://duckdb.org), an in-process SQL database that queries CSV files directly. Each query lives in its own file in [`sql/`](../sql), so it can also be run outside this notebook:

```bash
python scripts/run_sql.py sql/02_sentiment_by_airline.sql
```

| Query | Business question | SQL techniques |
|---|---|---|
| `01_data_quality.sql` | Can we trust the raw data? | CTEs, `GROUP BY … HAVING`, `LEFT JOIN`, `FILTER` |
| `02_sentiment_by_airline.sql` | Which airlines have the worst reputation? | conditional aggregation, `RANK()` |
| `03_complaint_drivers.sql` | Operations or customer service? | `CASE WHEN`, window `SUM() OVER (PARTITION BY …)` |
| `04_top_reason_per_airline.sql` | What should each airline fix first? | `ROW_NUMBER() OVER (PARTITION BY …)` |
| `05_daily_trend.sql` | Is sentiment getting better or worse? | `LAG()`, `RANK()`, date casting |
| `06_priority_matrix.sql` | Which airline + problem to fix first? | multiple CTEs, `JOIN`, index vs peers |

Queries 02–06 use the cleaned data in `data/processed/dashboard_data.csv`. The tests in `tests/test_sql.py` check that they return the same numbers as the dashboard.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from IPython.display import Markdown, display

from src.sql import SQL_DIR, run_sql_file


def show(name: str):
    """Print a query from sql/ and return its result."""
    path = SQL_DIR / name
    display(Markdown(f"```sql\n{path.read_text(encoding='utf-8').strip()}\n```"))
    return run_sql_file(path)

## 1. Data quality

Runs on the raw Kaggle file (`data/raw/Tweets.csv`), before any cleaning.

In [2]:
show("01_data_quality.sql")

```sql
-- Business question: Can we trust the raw data? Are tweets duplicated, and does
-- each tweet's airline label match the airline it is actually addressed to?
--
-- How to run (from the project root, needs data/raw/Tweets.csv from Kaggle):
--   python scripts/run_sql.py sql/01_data_quality.sql
--   or with the DuckDB CLI:  duckdb < sql/01_data_quality.sql
--
-- Reads the RAW file, before any cleaning. One row per airline label in the source.

WITH raw AS (
    SELECT * FROM read_csv_auto('data/raw/Tweets.csv')
),
duplicate_ids AS (
    -- tweet_ids that appear more than once, and whether their copies disagree
    SELECT tweet_id,
           ANY_VALUE(airline)                     AS airline,
           COUNT(*)                               AS copies,
           COUNT(DISTINCT airline_sentiment) > 1  AS conflicting_labels
    FROM raw
    GROUP BY tweet_id
    HAVING COUNT(*) > 1
),
duplicates_per_airline AS (
    SELECT airline,
           CAST(SUM(copies - 1) AS INTEGER)            AS duplicate_rows,
           COUNT(*) FILTER (WHERE conflicting_labels)  AS conflicting_ids
    FROM duplicate_ids
    GROUP BY airline
)
SELECT r.airline                                                                        AS source_label,
       COUNT(*)                                                                         AS rows,
       ANY_VALUE(COALESCE(d.duplicate_rows, 0))                                         AS duplicate_rows,
       ANY_VALUE(COALESCE(d.conflicting_ids, 0))                                        AS ids_with_conflicting_labels,
       ROUND(100.0 * COUNT(*) FILTER (WHERE r.text ILIKE '%@jetblue%') / COUNT(*), 1)  AS pct_mentioning_jetblue,
       ROUND(100.0 * COUNT(*) FILTER (WHERE r.text ILIKE '%@delta%')   / COUNT(*), 1)  AS pct_mentioning_delta
FROM raw AS r
LEFT JOIN duplicates_per_airline AS d USING (airline)
GROUP BY r.airline
ORDER BY rows DESC;
```

,source_label,rows,duplicate_rows,ids_with_conflicting_labels,pct_mentioning_jetblue,pct_mentioning_delta
0,United,3822,0,0,0.1,0.5
1,US Airways,2913,0,0,0.1,0.5
2,American,2759,155,18,0.3,0.7
3,Southwest,2420,0,0,0.1,0.5
4,Delta,2222,0,0,99.8,0.3
5,Virgin America,504,0,0,0.2,0.0


**Findings**

- **All 155 duplicate rows belong to American.** For 18 of those tweets the copies even carry different sentiment labels. The cleaning step keeps one copy per tweet, the one with the highest annotator confidence, which leaves 14,485 unique tweets.
- **The "Delta" label most likely refers to JetBlue.** 99.8% of the tweets labeled Delta (2,218 of 2,222) are addressed to @JetBlue, while only 0.3% mention @Delta. The source also has no JetBlue label at all, although JetBlue is one of the larger US airlines. Together this points to a labeling error in the source data.

  **Decision:** the analysis treats these tweets as JetBlue (see `AIRLINE_FIXES` in `src/data.py`). Results shown for JetBlue in this project rely on that assumption.

## 2. Airline reputation

In [3]:
show("02_sentiment_by_airline.sql")

```sql
-- Business question: Which airlines have the worst and best reputation on Twitter?
--
-- How to run (from the project root):
--   python scripts/run_sql.py sql/02_sentiment_by_airline.sql
--   or with the DuckDB CLI:  duckdb < sql/02_sentiment_by_airline.sql
--
-- Uses the cleaned data (duplicates removed, airline labels fixed).
-- net_sentiment_pp = positive share minus negative share, in percentage points.

WITH tweets AS (
    SELECT * FROM read_csv_auto('data/processed/dashboard_data.csv')
),
by_airline AS (
    SELECT airline,
           COUNT(*)                                        AS tweets,
           COUNT(*) FILTER (WHERE sentiment = 'negative')  AS negative,
           COUNT(*) FILTER (WHERE sentiment = 'neutral')   AS neutral,
           COUNT(*) FILTER (WHERE sentiment = 'positive')  AS positive
    FROM tweets
    GROUP BY airline
)
SELECT RANK() OVER (ORDER BY negative * 1.0 / tweets DESC)  AS rank_worst_first,
       airline,
       tweets,
       ROUND(100.0 * negative / tweets, 1)                  AS negative_pct,
       ROUND(100.0 * neutral  / tweets, 1)                  AS neutral_pct,
       ROUND(100.0 * positive / tweets, 1)                  AS positive_pct,
       ROUND(100.0 * (positive - negative) / tweets, 1)     AS net_sentiment_pp,
       ROUND(100.0 * negative / SUM(negative) OVER (), 1)   AS share_of_all_negative_pct
FROM by_airline
ORDER BY rank_worst_first;
```

,rank_worst_first,airline,tweets,negative_pct,neutral_pct,positive_pct,net_sentiment_pp,share_of_all_negative_pct
0,1,US Airways,2913,77.7,13.1,9.2,-68.5,24.9
1,2,American,2604,71.6,16.6,11.8,-59.8,20.5
2,3,United,3822,68.9,18.2,12.9,-56.0,29.0
3,4,Southwest,2420,49.0,27.4,23.6,-25.5,13.1
4,5,JetBlue,2222,43.0,32.5,24.5,-18.5,10.5
5,6,Virgin America,504,35.9,33.9,30.2,-5.8,2.0


**Findings**

- **US Airways has the worst reputation**: 77.7% of its tweets are negative, a net sentiment of −68.5 points. **Virgin America is the best** at −5.8 points.
- The worst *rate* is not the biggest *volume*: **United produces the most negative tweets (29.0% of all)** because it has the most tweets, even though its negative rate is third.

## 3. What drives complaints?

In [4]:
show("03_complaint_drivers.sql")

```sql
-- Business question: What drives customer complaints: operations (delays,
-- cancellations) or customer service?
--
-- How to run (from the project root):
--   python scripts/run_sql.py sql/03_complaint_drivers.sql
--   or with the DuckDB CLI:  duckdb < sql/03_complaint_drivers.sql
--
-- Groups the 10 annotated complaint reasons into 5 business areas and shows each
-- reason's share of all complaints next to its area's total share.

WITH tweets AS (
    SELECT * FROM read_csv_auto('data/processed/dashboard_data.csv')
),
complaints AS (
    SELECT negative_reason AS reason,
           CASE
               WHEN negative_reason IN ('Customer Service Issue', 'Flight Attendant Complaints') THEN 'Service'
               WHEN negative_reason IN ('Late Flight', 'Cancelled Flight')                    THEN 'Operations'
               WHEN negative_reason IN ('Lost Luggage', 'Damaged Luggage')                    THEN 'Baggage'
               WHEN negative_reason IN ('Bad Flight', 'Flight Booking Problems', 'Long Lines') THEN 'Product & booking'
               ELSE 'Unclear'
           END AS area
    FROM tweets
    WHERE sentiment = 'negative'
      AND negative_reason IS NOT NULL
),
by_reason AS (
    SELECT area, reason, COUNT(*) AS complaints
    FROM complaints
    GROUP BY area, reason
)
SELECT area,
       ROUND(100.0 * SUM(complaints) OVER (PARTITION BY area) / SUM(complaints) OVER (), 1) AS area_share_pct,
       reason,
       complaints,
       ROUND(100.0 * complaints / SUM(complaints) OVER (), 1)                              AS reason_share_pct
FROM by_reason
ORDER BY area_share_pct DESC, complaints DESC;
```

,area,area_share_pct,reason,complaints,reason_share_pct
0,Service,36.9,Customer Service Issue,2881,31.7
1,Service,36.9,Flight Attendant Complaints,475,5.2
2,Operations,27.3,Late Flight,1653,18.2
3,Operations,27.3,Cancelled Flight,830,9.1
4,Product & booking,14.0,Bad Flight,573,6.3
5,Product & booking,14.0,Flight Booking Problems,523,5.8
6,Product & booking,14.0,Long Lines,176,1.9
7,Unclear,13.0,Can't Tell,1180,13.0
8,Baggage,8.7,Lost Luggage,719,7.9
9,Baggage,8.7,Damaged Luggage,73,0.8


**Findings**

- **Service problems (36.9%) drive more complaints than operations (27.3%).** Customer Service Issue alone is 31.7% of complaints, more than late and cancelled flights combined.
- 13.0% of complaints are "Can't Tell": the annotators could not identify a reason. Queries 04 and 06 leave them out because an airline cannot act on them.

## 4. What should each airline fix first?

In [5]:
show("04_top_reason_per_airline.sql")

```sql
-- Business question: What are the #1 and #2 complaints for each airline, so each
-- airline knows what to fix first?
--
-- How to run (from the project root):
--   python scripts/run_sql.py sql/04_top_reason_per_airline.sql
--   or with the DuckDB CLI:  duckdb < sql/04_top_reason_per_airline.sql
--
-- "Can't Tell" is excluded: annotators could not identify a reason, so an airline
-- cannot act on it. Shares are of each airline's actionable complaints.

WITH tweets AS (
    SELECT * FROM read_csv_auto('data/processed/dashboard_data.csv')
),
counts AS (
    SELECT airline, negative_reason AS reason, COUNT(*) AS complaints
    FROM tweets
    WHERE sentiment = 'negative'
      AND negative_reason IS NOT NULL
      AND negative_reason <> 'Can''t Tell'
    GROUP BY airline, negative_reason
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY airline ORDER BY complaints DESC, reason)  AS reason_rank,
           ROUND(100.0 * complaints / SUM(complaints) OVER (PARTITION BY airline), 1) AS share_of_airline_pct
    FROM counts
)
SELECT airline, reason_rank, reason, complaints, share_of_airline_pct
FROM ranked
WHERE reason_rank <= 2
ORDER BY airline, reason_rank;
```

,airline,reason_rank,reason,complaints,share_of_airline_pct
0,American,1,Customer Service Issue,739,44.1
1,American,2,Late Flight,237,14.1
2,JetBlue,1,Late Flight,269,35.0
3,JetBlue,2,Customer Service Issue,199,25.9
4,Southwest,1,Customer Service Issue,391,38.1
5,Southwest,2,Cancelled Flight,162,15.8
6,US Airways,1,Customer Service Issue,811,40.2
7,US Airways,2,Late Flight,453,22.5
8,United,1,Customer Service Issue,681,30.2
9,United,2,Late Flight,525,23.3


**Findings**

- **Customer Service Issue is the #1 complaint at 5 of 6 airlines**, between 30% and 44% of each airline's actionable complaints (American is highest at 44.1%).
- **JetBlue is the exception: its #1 complaint is Late Flight (35.0%).** For JetBlue, punctuality matters more than service.

## 5. Daily trend

In [6]:
show("05_daily_trend.sql")

```sql
-- Business question: Is sentiment getting better or worse from day to day, and
-- which day was worst for each airline?
--
-- How to run (from the project root):
--   python scripts/run_sql.py sql/05_daily_trend.sql
--   or with the DuckDB CLI:  duckdb < sql/05_daily_trend.sql
--
-- Airline-days with fewer than 30 tweets are dropped (same rule as the dashboard):
-- a share from a handful of tweets is noise. Times are US Pacific.
-- change_pp = change in negative share vs the airline's previous day shown.

WITH tweets AS (
    SELECT * FROM read_csv_auto('data/processed/dashboard_data.csv')
),
daily AS (
    SELECT airline,
           CAST(tweet_created AS DATE)                                       AS day,
           COUNT(*)                                                          AS tweets,
           100.0 * COUNT(*) FILTER (WHERE sentiment = 'negative') / COUNT(*) AS negative_pct
    FROM tweets
    GROUP BY airline, CAST(tweet_created AS DATE)
    HAVING COUNT(*) >= 30
)
SELECT airline,
       day,
       tweets,
       ROUND(negative_pct, 1)                                                              AS negative_pct,
       ROUND(negative_pct - LAG(negative_pct) OVER (PARTITION BY airline ORDER BY day), 1) AS change_pp,
       RANK() OVER (PARTITION BY airline ORDER BY negative_pct DESC)                       AS worst_day_rank
FROM daily
ORDER BY airline, day;
```

,airline,day,tweets,negative_pct,change_pp,worst_day_rank
0,American,2015-02-22,982,77.1,NaN,1
1,American,2015-02-23,1139,72.3,-4.7,2
2,American,2015-02-24,479,58.7,-13.7,3
3,JetBlue,2015-02-17,263,41.1,NaN,3
4,JetBlue,2015-02-18,268,39.2,-1.9,5
5,JetBlue,2015-02-19,283,47.7,8.5,2
6,JetBlue,2015-02-20,251,36.3,-11.4,6
7,JetBlue,2015-02-21,243,40.3,4.1,4
8,JetBlue,2015-02-22,408,62.5,22.2,1
9,JetBlue,2015-02-23,391,32.0,-30.5,8


**Findings**

- **Sentiment got worse at the weekend of 21–22 Feb.** 22 Feb was the worst day of the week for United (75.7% negative), US Airways (86.6%), JetBlue (62.5%) and Virgin America (50.9%). Southwest peaked one day earlier, on 21 Feb (69.5%).
- The data does not say why. The next step would be to match these days against operational data (weather, cancellations) for the same dates.
- American has only 4 tweets before 22 Feb, so its trend cannot be compared with the others.

## 6. Priority matrix

In [7]:
show("06_priority_matrix.sql")

```sql
-- Business question: Which airline + complaint combinations should the Customer
-- Experience team fix first?
--
-- How to run (from the project root):
--   python scripts/run_sql.py sql/06_priority_matrix.sql
--   or with the DuckDB CLI:  duckdb < sql/06_priority_matrix.sql
--
-- For each airline and complaint reason:
--   complaints      number of negative tweets (size of the problem)
--   pct_of_all      share of ALL actionable complaints in the data (impact)
--   index_vs_peers  the reason's share of this airline's complaints divided by its
--                   share across all airlines. 1.5 = 50% more common than at other
--                   airlines, so it is an airline-specific problem, not an industry one.
-- "Can't Tell" is excluded (not actionable). Top 10 by number of complaints.

WITH tweets AS (
    SELECT * FROM read_csv_auto('data/processed/dashboard_data.csv')
),
complaints AS (
    SELECT airline, negative_reason AS reason
    FROM tweets
    WHERE sentiment = 'negative'
      AND negative_reason IS NOT NULL
      AND negative_reason <> 'Can''t Tell'
),
pair AS (
    SELECT airline, reason, COUNT(*) AS complaints
    FROM complaints
    GROUP BY airline, reason
),
airline_total AS (
    SELECT airline, COUNT(*) AS airline_complaints
    FROM complaints
    GROUP BY airline
),
reason_overall AS (
    SELECT reason, COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS overall_share
    FROM complaints
    GROUP BY reason
)
SELECT p.airline,
       p.reason,
       p.complaints,
       ROUND(100.0 * p.complaints / SUM(p.complaints) OVER (), 1)             AS pct_of_all,
       ROUND(p.complaints * 1.0 / a.airline_complaints / r.overall_share, 2)  AS index_vs_peers
FROM pair AS p
JOIN airline_total  AS a USING (airline)
JOIN reason_overall AS r USING (reason)
ORDER BY p.complaints DESC
LIMIT 10;
```

,airline,reason,complaints,pct_of_all,index_vs_peers
0,US Airways,Customer Service Issue,811,10.3,1.10
1,American,Customer Service Issue,739,9.4,1.21
2,United,Customer Service Issue,681,8.6,0.83
3,United,Late Flight,525,6.6,1.11
4,US Airways,Late Flight,453,5.7,1.07
5,Southwest,Customer Service Issue,391,4.9,1.04
6,JetBlue,Late Flight,269,3.4,1.67
7,United,Lost Luggage,269,3.4,1.31
8,American,Late Flight,237,3.0,0.68
9,American,Cancelled Flight,229,2.9,1.30


**Findings**

- **Customer service at US Airways and American alone accounts for 19.7% of all actionable complaints.** These two are the highest-impact fixes in the data.
- `index_vs_peers` shows problems specific to one airline: **JetBlue's late flights (1.67×)**, **United's lost luggage (1.31×)** and **American's cancellations (1.30×)** are much more common than at other airlines, so they need an airline-specific fix rather than an industry benchmark.

## Conclusion

| Question | Answer from SQL |
|---|---|
| Worst / best reputation | US Airways (77.7% negative) / Virgin America (35.9%) |
| Biggest complaint driver | Service (36.9% of complaints), ahead of operations (27.3%) |
| What to fix first | Customer service at US Airways and American (19.7% of actionable complaints) |
| Airline-specific problems | JetBlue delays, United lost luggage, American cancellations |
| Trend | Negativity peaked on 21–22 Feb for every airline with a full week of data |

The SQL results match the pandas analysis in notebook 01 and the dashboard (checked by `tests/test_sql.py`), and add three insights the dashboard does not show directly: the per-airline #1 complaint, the weekend peak, and the airline-specific problems.